# 🧠 Алгоритми та структури даних · Урок 4 — Сортування та пошук

Сортування — найкласичніша тема курсу алгоритмів: на ній вчаться аналізу складності, парадигмам
(«розділяй і володарюй») і компромісам. Розберемо пошук (лінійний і двійковий) та всі головні
алгоритми сортування — від «шкільних» O(n²) до O(n log n) і лінійних.

**Передумова:** [Урок 1 — складність](lektsiya-1-skladnist-ta-rekursiya.ipynb).

### Що ви вмітимете
- реалізувати **двійковий пошук** і розуміти його інваріант;
- знати прості (bubble/selection/insertion) і ефективні (merge/quick/heap) сортування;
- розуміти **стабільність**, нижню межу **O(n log n)** і чому Python використовує **Timsort**.

## 1. Пошук

### Лінійний пошук — O(n)
Перебираємо елементи підряд, доки не знайдемо. Працює на **будь-якому** масиві.

### Двійковий пошук — O(log n)
Працює **лише на відсортованому** масиві. Дивимось на **середній** елемент і щоразу відкидаємо
**половину**: якщо шуканий менший — шукаємо ліворуч, більший — праворуч.

```
шукаємо 7 у [1, 3, 5, 7, 9, 11]:
  mid=5 -> 7>5 -> праворуч [7, 9, 11]
  mid=9 -> 7<9 -> ліворуч  [7]
  mid=7 -> знайшли!
```

In [ ]:
def binary_search(arr, target):
    lo, hi = 0, len(arr) - 1          # інваріант: якщо target є, він у [lo, hi]
    while lo <= hi:
        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid                # індекс знайденого
        elif arr[mid] < target:
            lo = mid + 1              # відкидаємо ліву половину
        else:
            hi = mid - 1              # відкидаємо праву половину
    return -1                         # не знайдено

data = [1, 3, 5, 7, 9, 11]
print("індекс 7:", binary_search(data, 7))    # 3
print("індекс 8:", binary_search(data, 8))    # -1

# У стандартній бібліотеці двійковий пошук — модуль bisect:
import bisect
print("bisect_left(8):", bisect.bisect_left(data, 8))   # 4 — куди вставити, зберігши порядок

## 2. Прості сортування — O(n²)

«Шкільні» алгоритми: прості, наочні, але повільні на великих даних. Усі — O(n²) у середньому.

- **Bubble sort (бульбашкове)** — сусідні елементи міняються місцями, «важчі» спливають у кінець.
- **Selection sort (вибором)** — шукаємо мінімум і ставимо на початок, потім наступний, і т.д.
- **Insertion sort (вставками)** — беремо елемент і «вставляємо» його на правильне місце серед
  уже відсортованих ліворуч (як розкладають карти в руці). На **майже впорядкованих** — швидко (~O(n)).

In [ ]:
def bubble_sort(a):
    a = a[:]                                  # копія, щоб не псувати вхід
    n = len(a)
    for i in range(n):
        swapped = False
        for j in range(n - 1 - i):            # найбільший «спливає» в кінець
            if a[j] > a[j + 1]:
                a[j], a[j + 1] = a[j + 1], a[j]
                swapped = True
        if not swapped:                       # уже відсортовано -> вихід (best case O(n))
            break
    return a

def selection_sort(a):
    a = a[:]
    for i in range(len(a)):
        m = i
        for j in range(i + 1, len(a)):        # шукаємо мінімум у решті
            if a[j] < a[m]:
                m = j
        a[i], a[m] = a[m], a[i]               # ставимо його на місце i
    return a

def insertion_sort(a):
    a = a[:]
    for i in range(1, len(a)):
        key = a[i]
        j = i - 1
        while j >= 0 and a[j] > key:          # зсуваємо більші праворуч
            a[j + 1] = a[j]
            j -= 1
        a[j + 1] = key                        # вставляємо key на місце
    return a

data = [5, 2, 9, 1, 5, 6]
print("bubble   :", bubble_sort(data))
print("selection:", selection_sort(data))
print("insertion:", insertion_sort(data))

## 3. Ефективні сортування — O(n log n)

### Merge sort (злиттям) — «розділяй і володарюй»
Ділимо масив навпіл, **рекурсивно** сортуємо кожну половину, потім **зливаємо** дві відсортовані
частини в одну. Завжди **O(n log n)**, **стабільний**, але потребує **O(n)** додаткової пам'яті.

### Quick sort (швидке)
Обираємо **опорний** елемент (pivot), розбиваємо масив на «менші за pivot» і «більші», рекурсивно
сортуємо частини. У середньому **O(n log n)** і дуже швидкий на практиці, але **найгірший — O(n²)**
(поганий вибір pivot на вже відсортованих даних; лікується випадковим pivot).

### Heap sort (купою)
Будуємо купу (Урок 3) і по черзі дістаємо мінімум/максимум. **O(n log n)**, пам'ять **O(1)**, але
не стабільний.

In [ ]:
def merge_sort(a):
    if len(a) <= 1:
        return a[:]
    mid = len(a) // 2
    left = merge_sort(a[:mid])            # рекурсія на половинах
    right = merge_sort(a[mid:])
    return merge(left, right)

def merge(left, right):
    out, i, j = [], 0, 0
    while i < len(left) and j < len(right):    # зливаємо дві відсортовані частини
        if left[i] <= right[j]:                # <= робить сортування СТАБІЛЬНИМ
            out.append(left[i]); i += 1
        else:
            out.append(right[j]); j += 1
    out.extend(left[i:]); out.extend(right[j:])
    return out

def quick_sort(a):
    if len(a) <= 1:
        return a[:]
    pivot = a[len(a) // 2]                      # опорний (для простоти — середній)
    less = [x for x in a if x < pivot]
    equal = [x for x in a if x == pivot]
    greater = [x for x in a if x > pivot]
    return quick_sort(less) + equal + quick_sort(greater)

data = [5, 2, 9, 1, 5, 6, 3]
print("merge:", merge_sort(data))
print("quick:", quick_sort(data))

## 4. Лінійні сортування — O(n) (без порівнянь)

Якщо про дані відомо **більше** (напр. це малі цілі числа), можна обійти межу O(n log n):

- **Counting sort (підрахунком)** — рахуємо, скільки разів трапляється кожне значення, і
  відтворюємо масив. **O(n + k)**, де `k` — діапазон значень. Годиться, коли `k` невеликий.
- **Radix sort (порозрядне)** — сортує числа по розрядах (одиниці, десятки…), використовуючи
  counting sort для кожного розряду. **O(d·(n + k))**.
- **Bucket sort (кошиками)** — розкидає елементи по «кошиках» за діапазонами, сортує кожен окремо.

⚠️ Вони не універсальні: працюють лише за певних умов (обмежений діапазон, числові ключі).

In [ ]:
def counting_sort(a):
    if not a:
        return []
    lo, hi = min(a), max(a)
    counts = [0] * (hi - lo + 1)          # комірка на кожне можливе значення
    for x in a:
        counts[x - lo] += 1               # рахуємо входження
    out = []
    for i, c in enumerate(counts):        # відтворюємо у відсортованому порядку
        out.extend([i + lo] * c)
    return out

data = [4, 2, 2, 8, 3, 3, 1]
print("counting sort:", counting_sort(data))   # O(n + k), без жодного порівняння

## 5. Порівняльна таблиця

| Алгоритм | Середній | Найгірший | Пам'ять | Стабільний? |
|----------|----------|-----------|---------|-------------|
| Bubble | O(n²) | O(n²) | O(1) | ✅ |
| Selection | O(n²) | O(n²) | O(1) | ❌ |
| Insertion | O(n²) | O(n²) | O(1) | ✅ |
| **Merge** | O(n log n) | O(n log n) | O(n) | ✅ |
| **Quick** | O(n log n) | **O(n²)** | O(log n) | ❌ |
| **Heap** | O(n log n) | O(n log n) | O(1) | ❌ |
| Counting | O(n+k) | O(n+k) | O(k) | ✅ |

> 🔎 **Важливо знати — нижня межа O(n log n).** Будь-яке сортування, що спирається **лише на
> порівняння** елементів, не може бути швидшим за **O(n log n)** у найгіршому разі (це доводять
> через «дерево рішень»: n! можливих порядків потребують ≥ log(n!) ≈ n log n порівнянь). Обійти цю
> межу можна тільки **не порівнюючи** (counting/radix), використавши додаткові знання про дані.

## 6. 🔎 Стабільність і Timsort у Python

**Стабільне** сортування зберігає **відносний порядок рівних** елементів. Це важливо при
сортуванні за кількома ключами (напр. спершу за іменем, потім стабільно за віком — імена в межах
одного віку лишаться впорядкованими).

Вбудовані `sorted()` та `list.sort()` у Python використовують **Timsort** — гібрид **merge sort +
insertion sort**, який:
- **стабільний**;
- гарантує **O(n log n)** у найгіршому разі;
- дуже швидкий на **частково впорядкованих** даних (шукає готові «серії» — runs).

In [ ]:
# Стабільність: сортуємо пари (ім'я, вік) за віком. Рівні за віком зберігають вихідний порядок.
people = [("Іван", 30), ("Олег", 25), ("Анна", 30), ("Марія", 25)]

by_age = sorted(people, key=lambda p: p[1])    # sorted стабільний
print("за віком (стабільно):")
for name, age in by_age:
    print(f"  {age}: {name}")
# 25: Олег, 25: Марія (Олег перед Марією — як у вході), потім 30: Іван, 30: Анна

# key дозволяє сортувати за чим завгодно:
print("\nза довжиною імені:", sorted(["bb", "a", "ccc"], key=len))
print("за спаданням     :", sorted([3, 1, 2], reverse=True))

# ✅ Підсумок уроку
- **Двійковий пошук** — O(log n), але лише на **відсортованому** масиві (відкидаємо половину щокроку).
- **Прості** (bubble/selection/insertion) — O(n²); insertion гарний на майже впорядкованих.
- **Ефективні** — O(n log n): **merge** (стабільний, +O(n) пам'яті), **quick** (швидкий, але worst O(n²)), **heap** (пам'ять O(1)).
- **Лінійні** (counting/radix/bucket) — O(n) за спец-умов (обмежений діапазон), без порівнянь.
- **Нижня межа** порівняльних сортувань — **O(n log n)**.
- Python `sorted`/`sort` = **Timsort**: стабільний, O(n log n), швидкий на впорядкованих даних; `key`/`reverse` — головні інструменти.

### ▶️ Далі
Урок 5 — **графи та алгоритми на них**.

### 📚 Хочу знати більше
- Візуалізація сортувань: <https://visualgo.net/en/sorting>
- Timsort: <https://en.wikipedia.org/wiki/Timsort>
- `sorted()` HOWTO: <https://docs.python.org/3/howto/sorting.html>